## Phase 3 - Isolation Forest modelling
**Goal:** Scale the features and drop unwanted colums like annotation samples and symbols.

In [1]:
import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

C:\Users\Manoj\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# Load record
record = wfdb.rdrecord('100', pn_dir='mitdb')
annotation = wfdb.rdann('100', 'atr', pn_dir='mitdb')

In [3]:
ecg_signal = record.p_signal[:, 0]
fs = record.fs
ann_samples = annotation.sample
ann_symbols = annotation.symbol

In [4]:
# Rebuild feature DataFrame
rr_intervals = np.diff(ann_samples)
rr_ms = (rr_intervals / fs) * 1000
heart_rate = 60000 / rr_ms

feature_df = pd.DataFrame({
    'sample': ann_samples[1:],
    'symbol': ann_symbols[1:],
    'rr_interval': rr_ms,
    'heart_rate': heart_rate,
    'rr_diff': np.append(0, np.diff(rr_ms)),
})

print(f"Feature DataFrame ready - shape: {feature_df.shape}")

Feature DataFrame ready - shape: (2273, 5)


In [5]:
# Select only the numeric features for the model
X = feature_df[['rr_interval', 'heart_rate', 'rr_diff']].values

### Why Scaling?
Our three features are on very different scales 
- RR interval is in hundreds of ms 
- rr_diff can be negative 
- heart rate is in tens of bpm 

Without scaling, the model would unfairly weight whichever feature has the largest numbers

In [6]:
# Scale - Isolation Forest works better when features are on the same scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [7]:
print("Features scaled successfully")
print(f"Mean after scaling (should be ~0): {X_scaled.mean(axis=0).round(3)}")
print(f"Std after scaling  (should be ~1): {X_scaled.std(axis=0).round(3)}")

Features scaled successfully
Mean after scaling (should be ~0): [ 0.  0. -0.]
Std after scaling  (should be ~1): [1. 1. 1.]


### Scaling Verification

Mean ≈ 0 and Std ≈ 1 confirmed for all three features.
All features are now on equal footing - the model will judge 
rr_interval, heart_rate and rr_diff with equal weight.

### Model Training
Ensure we have right amount of trees (~100) so that we avoid overfitting sometimes, reduce computational power & resources and explicitly mention what's the anomaly ratio seen in the data inputs.

In [8]:
# Train the model
# contamination = expected proportion of anomalies (we know ~1.5% are abnormal)
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.015,
    random_state=42
)

iso_forest.fit(X_scaled)

IsolationForest(contamination=0.015, random_state=42)

In [9]:
# Predict - Isolation Forest returns: -1 = anomaly, 1 = normal
predictions = iso_forest.predict(X_scaled)

In [10]:
# Add predictions to DataFrame
feature_df['anomaly'] = predictions
feature_df['anomaly_flag'] = (predictions == -1).astype(int)

In [11]:
print(f"Total beats: {len(feature_df)}")
print(f"Flagged anomalies: {feature_df['anomaly_flag'].sum()}")
print(f"Anomaly rate: {feature_df['anomaly_flag'].mean()*100:.2f}%")

Total beats: 2273
Flagged anomalies: 35
Anomaly rate: 1.54%


### Model Evaluation
Out of all the given input data points, how many of the flagged anomalies were actually abnormal beats?

In [12]:
print("*** Anomaly Detection Results ***")
print("Actual beat distribution:")
print(feature_df['symbol'].value_counts())
print()
print("Flagged beats by type:")
flagged = feature_df[feature_df['anomaly_flag'] == 1]
print(flagged['symbol'].value_counts())

*** Anomaly Detection Results ***
Actual beat distribution:
symbol
N    2239
A      33
V       1
Name: count, dtype: int64

Flagged beats by type:
symbol
N    20
A    14
V     1
Name: count, dtype: int64


In [13]:
# What % of true abnormal beats did we catch?
true_abnormal = feature_df[feature_df['symbol'] != 'N']
caught = true_abnormal[true_abnormal['anomaly_flag'] == 1]
print(f"True abnormal beats: {len(true_abnormal)}")
print(f"Caught by model: {len(caught)}")
print(f"Detection rate: {len(caught)/len(true_abnormal)*100:.1f}%")

True abnormal beats: 34
Caught by model: 15
Detection rate: 44.1%


### Key take aways
**What the model flagged (35 total):**
- 14 A beats (Premature Atrial Contractions) - genuinely abnormal 
- 1 V beat (Premature Ventricular Contraction) - genuinely abnormal 
- 20 N beats (Normal) - incorrectly flagged 

**Detection rate: 44.1%** - the model caught 15 out of 34 true abnormal beats.

**What this means:**
- The model missed 19 abnormal beats (false negatives)
- It incorrectly flagged 20 normal beats (false positives)
- 44.1% recall on a first unsupervised attempt with 3 features is a baseline, not a failure

**Why didn't it catch more?**
Many of the missed A beats have RR intervals and heart rates that overlap 
with the normal range. Their rhythm deviation is subtle - not extreme enough 
to stand out as a statistical outlier with only 3 features.